# 02 — Train on your own grid

A template: edit the **settings** cell and run all cells. The defaults use the toy grid written by
notebook 01, so the template runs as it is.

Your grid file (`.pt` or `.npz`) must contain one entry per simulation for every key:
* each parameter as shape `[N]`;
* the simulated data, any shape `[N, ...]` (e.g. `[N, n_wavelengths]`);
* optionally context values, e.g. `[N]`.

Data that should be noisy must be noise-free in the grid if you add noise with an augmentation step,
or already noisy if you do not.

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():  # Running from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from sbi_nre import *

DEVICE = ('cuda' if torch.cuda.is_available()          # NVIDIA GPU
          else 'mps' if torch.backends.mps.is_available()  # Apple silicon GPU
          else 'cpu')
print('Device:', DEVICE)

## Settings

In [ ]:
GRID_PATH = ROOT / 'data' / 'toy_grid.pt'
SPLIT = (0.8, 0.1, 0.1)  # training, validation, test

# What to infer: any 1D or 2D parameter groups.
PARAM_GROUPS = [('depth', 'width'), ('centre',), ('slope',)]

# Augmentation (training only) and preprocessing (training and observations).
AUGMENT = [step('add_noise', key='signal', out='obs', sigma_key='sigma', sigma_range=[0.005, 0.05])]
PREPROCESS = [step('normalise_minmax', key='obs')]

# Network inputs (keys after augmentation and preprocessing).
OBSERVATION = ['obs_norm', 'obs_min', 'obs_max']
CONTEXT = ['sigma']
NETWORK = {'summary_size': 64, 'hidden': [256, 256], 'tail_hidden': [128, 128]}

# Training
RUN_NAME = 'my_first_run'
TRAINING = {'batch_size': 256, 'learning_rate': 1e-3, 'max_epochs': 50, 'patience': 5}

### Optional: your own transform

Any function `data -> data` can become a step. Register it with `@transform`, then use it in `AUGMENT` or
`PREPROCESS` by name. Run this cell again before loading a run that uses it. Example: subtract each
spectrum's mean (uncomment to use).

In [ ]:
@transform
def subtract_mean(data, *, key='obs'):
    return {**data, f'{key}_centred': data[key] - data[key].mean(dim=-1, keepdim=True)}

# PREPROCESS = PREPROCESS + [step('subtract_mean', key='obs')]
# OBSERVATION = OBSERVATION + ['obs_centred']

## 1. Load, split and check the inputs

In [ ]:
grid = load_grid(GRID_PATH)
training, validation, test = split(grid, SPLIT, seed=0)
example = apply(PREPROCESS, apply(AUGMENT, select(training, slice(0, 4))))
print('Simulations:', n_samples(grid))
for key, value in example.items():
    print(f'{key:>10}: shape {tuple(value.shape[1:]) or "scalar"}', '(input)' if key in OBSERVATION + CONTEXT else '')

## 2. Build and train

In [ ]:
model = build_model(example, PARAM_GROUPS, OBSERVATION, CONTEXT, **NETWORK)
history = train(model, training, validation, ROOT / 'runs' / RUN_NAME, augment=AUGMENT,
                preprocess=PREPROCESS, device=DEVICE, **TRAINING)
plot_history(history);

## 3. Posterior for one observation

For a real observation, load it into a dictionary with the same raw keys as the grid *after
augmentation* (here `obs` and `sigma`), e.g. `observation = {'obs': torch.tensor(...), 'sigma': 0.02}`.
Here we use a noisy test simulation instead, so we know the truth.

The prior samples are the validation simulations' parameters. If your prior is uniform with known ranges,
`uniform_prior_samples({'param': (low, high), ...}, 200_000)` gives many more, and a smoother posterior.

In [ ]:
run = load_run(ROOT / 'runs' / RUN_NAME, device=DEVICE)
observation = select(apply(AUGMENT, select(test, slice(0, 1))), 0)
truths = {p: float(observation[p]) for p in run.model.param_names}
result = posterior(run, observation, prior_samples=validation)
print('Effective sample size:', {name: round(effective_sample_size(g['weights'])) for name, g in result.items()})
plot_posterior(result, truths);
summarise(result)

## 4. Check on test simulations: true value against median, and coverage

In [ ]:
N_CHECK = 200
noisy_test = apply(AUGMENT, select(test, slice(0, N_CHECK)))
params = run.model.param_names
medians, truths = {p: [] for p in params}, {p: noisy_test[p].tolist() for p in params}
for i in range(N_CHECK):
    summary = summarise(posterior(run, select(noisy_test, i), prior_samples=validation))
    for p in params:
        medians[p].append(summary[p][1])
plot_true_vs_median(truths, medians);
for p, fractions in coverage(run, noisy_test, validation).items():
    print(f'{p:>10}: ' + '   '.join(f'{level:.0%} interval contains truth in {f:.0%}' for level, f in fractions.items()))